# Module 2.1: ART Architecture Walkthrough

## Learning Objectives

By completing this notebook, you will:
1. Understand ART's client-backend architecture
2. Configure a training run with model, GRPO, and RULER settings
3. Build and inspect trajectory data structures
4. Estimate training costs for different model sizes

## Estimated Time: 15 minutes

---

## 1. ART Architecture

ART splits into two parts:

```
┌─────────────────────┐     ┌──────────────────────────────┐
│      CLIENT          │     │          BACKEND              │
│                      │     │                                │
│  Your agent code     │────>│  vLLM Inference Service       │
│  Tool calls          │<────│  (fast model responses)       │
│  Trajectory recording│     │                                │
│                      │     │  Unsloth GRPO Trainer          │
│                      │     │  (weight updates + LoRA swap)  │
└─────────────────────┘     └──────────────────────────────┘
```

The **client** sends inference requests and records trajectories.
The **backend** generates responses and trains the model.

In [ ]:
# ART configuration in Python
# This mirrors what you'd pass to ART's initialization

from dataclasses import dataclass, field


@dataclass
class ARTConfig:
    """Complete ART training configuration."""
    # Model
    model_name: str = "Qwen/Qwen2.5-3B"
    max_seq_length: int = 4096
    
    # LoRA
    lora_rank: int = 16
    lora_alpha: int = 32
    
    # GRPO
    group_size: int = 4
    learning_rate: float = 2e-5
    kl_coeff: float = 0.05
    clip_epsilon: float = 0.2
    num_steps: int = 50
    
    # RULER
    judge_model: str = "openai/o4-mini"
    
    # Infrastructure
    gpu_memory_utilization: float = 0.85
    mcp_server_url: str = "http://localhost:8000"


config = ARTConfig()
print(f"Model: {config.model_name}")
print(f"GRPO group size: {config.group_size}")
print(f"Training steps: {config.num_steps}")
print(f"Judge: {config.judge_model}")

## 2. Trajectories

A **trajectory** is the complete history of one agent run:
every message, tool call, and tool result.

This is what GRPO learns from — the full chain of decisions.

In [ ]:
# A trajectory is a list of messages
# Here's what a real multi-turn trajectory looks like

trajectory_good = [
    {"role": "system", "content": "You are a SQL expert. Use tools to answer questions."},
    {"role": "user", "content": "Who leads the most expensive active project?"},
    
    # Agent decides to explore schema first
    {"role": "assistant", "content": None, "tool_calls": [
        {"id": "call_1", "function": {"name": "list_tables", "arguments": "{}"}}
    ]},
    {"role": "tool", "tool_call_id": "call_1", "content": "departments\nemployees\nprojects"},
    
    # Agent checks project schema
    {"role": "assistant", "content": None, "tool_calls": [
        {"id": "call_2", "function": {"name": "describe_table", "arguments": '{"table_name": "projects"}'}}
    ]},
    {"role": "tool", "tool_call_id": "call_2", "content": "id INTEGER PK\nname TEXT\nlead_id INTEGER\nbudget REAL\nstatus TEXT"},
    
    # Agent writes correct JOIN query
    {"role": "assistant", "content": None, "tool_calls": [
        {"id": "call_3", "function": {"name": "run_query", "arguments": '{"sql": "SELECT e.name, p.name, p.budget FROM projects p JOIN employees e ON p.lead_id = e.id WHERE p.status = \\"active\\" ORDER BY p.budget DESC LIMIT 1"}'}}
    ]},
    {"role": "tool", "tool_call_id": "call_3", "content": "Iris Nakamura | Data Warehouse Migration | 600000"},
    
    # Final answer
    {"role": "assistant", "content": "Iris Nakamura leads the most expensive active project: Data Warehouse Migration with a budget of $600,000."},
]

trajectory_bad = [
    {"role": "system", "content": "You are a SQL expert. Use tools to answer questions."},
    {"role": "user", "content": "Who leads the most expensive active project?"},
    
    # Agent skips schema exploration, guesses column names
    {"role": "assistant", "content": None, "tool_calls": [
        {"id": "call_1", "function": {"name": "run_query", "arguments": '{"sql": "SELECT leader FROM project WHERE active = 1 ORDER BY cost DESC LIMIT 1"}'}}
    ]},
    {"role": "tool", "tool_call_id": "call_1", "content": "SQL Error: no such table: project"},
    
    # Agent hallucinates instead of retrying
    {"role": "assistant", "content": "The most expensive active project is led by John Smith."},
]

print(f"Good trajectory: {len(trajectory_good)} messages, 3 tool calls")
print(f"Bad trajectory:  {len(trajectory_bad)} messages, 1 tool call (failed)")
print()
print("Good trajectory explored schema → wrote correct JOIN → accurate answer")
print("Bad trajectory guessed columns → SQL error → hallucinated answer")

## 3. How GRPO Learns from Trajectories

For each scenario, ART generates multiple trajectories (group_size=4).
RULER scores them. GRPO reinforces the good ones.

In [ ]:
import numpy as np

# Simulated: 4 trajectories for the same question
trajectory_descriptions = [
    "Explored schema, wrote correct JOIN, accurate answer",
    "Explored schema, wrong WHERE clause, partial answer",
    "No schema check, SQL error, hallucinated answer",
    "Explored schema, correct query, but answered with wrong number",
]

# RULER scores (from LLM judge)
rewards = np.array([0.95, 0.5, 0.1, 0.4])

# GRPO advantage computation
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-8)

print("GRPO Training Signal")
print("=" * 60)
for desc, reward, adv in zip(trajectory_descriptions, rewards, advantages):
    action = "REINFORCE" if adv > 0 else "SUPPRESS "
    print(f"  [{action}] reward={reward:.2f}, advantage={adv:+.2f}")
    print(f"    {desc}")
    print()

## 4. Cost Estimation

How much does it cost to train an agent with ART?

In [ ]:
def estimate_training_cost(
    model_size_b: float,
    num_steps: int,
    group_size: int,
    scenarios: int,
    cost_per_gpu_hour: float = 2.0,
) -> dict:
    """
    Estimate ART training cost.
    
    Based on ART benchmarks:
    - 3B model: ~0.5 GPU-hours per 50 steps
    - 14B model: ~4 GPU-hours per 50 steps
    - Linear scaling with group_size and steps
    """
    # Base hours per 50 steps (from benchmarks)
    base_hours_per_50 = 0.5 * (model_size_b / 3)  # rough linear scaling
    
    step_factor = num_steps / 50
    group_factor = group_size / 4
    
    gpu_hours = base_hours_per_50 * step_factor * group_factor
    training_cost = gpu_hours * cost_per_gpu_hour
    
    # RULER judge costs (approx $0.01 per scenario per step)
    judge_cost = scenarios * num_steps * 0.01
    
    total = training_cost + judge_cost
    
    return {
        "gpu_hours": round(gpu_hours, 1),
        "training_cost": round(training_cost, 2),
        "judge_cost": round(judge_cost, 2),
        "total_cost": round(total, 2),
    }


# Compare model sizes
for size in [3, 7, 14]:
    est = estimate_training_cost(
        model_size_b=size, num_steps=100, group_size=4, scenarios=20
    )
    print(f"Qwen2.5-{size}B: {est['gpu_hours']}h GPU, ${est['total_cost']:.2f} total")

## Exercise: Build Your Own Config

**Task:** Create an ART config for training a 7B model on a customer support task.

Requirements:
- Use Qwen/Qwen2.5-7B
- Group size of 6 (more diverse trajectories)
- 80 training steps
- LoRA rank of 32 (higher capacity)
- Estimate the training cost

In [ ]:
# YOUR CODE HERE
# ---------------

# Create a config with the requirements above
my_config = ARTConfig(
    # TODO: fill in the parameters
)

# Estimate cost
my_cost = estimate_training_cost(
    # TODO: fill in parameters from your config
    model_size_b=0,
    num_steps=0,
    group_size=0,
    scenarios=20,
)

exercise_config = my_config  # Don't change this line
exercise_cost = my_cost      # Don't change this line

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise():
    assert exercise_config.model_name == "Qwen/Qwen2.5-7B", \
        f"Expected Qwen2.5-7B, got {exercise_config.model_name}"
    assert exercise_config.group_size == 6, \
        f"Expected group_size=6, got {exercise_config.group_size}"
    assert exercise_config.num_steps == 80, \
        f"Expected 80 steps, got {exercise_config.num_steps}"
    assert exercise_config.lora_rank == 32, \
        f"Expected lora_rank=32, got {exercise_config.lora_rank}"
    assert exercise_cost["total_cost"] > 0, \
        "Cost estimate should be positive"
    print("All tests passed!")
    print(f"Estimated cost: ${exercise_cost['total_cost']:.2f}")

test_exercise()

## Summary

**Key Takeaways:**
1. ART splits into Client (your code) and Backend (vLLM + Unsloth)
2. Trajectories record the full history of agent decisions
3. GRPO learns from groups of trajectories: reinforce good, suppress bad
4. Training cost is modest: a 14B model for under $80

**Next:** Module 03 covers RULER automatic reward functions.